# Cybershuttle SDK -  Molecular Dynamics
> Define, run, monitor, and analyze molecular dynamics experiments in a HPC-agnostic way.

This notebook shows how users can setup and launch a **NAMD** experiment with replicas, monitor its execution, and run analyses both during and after execution.

## Installing Required Packages

First, install the `airavata-python-sdk-test` package from the pip repository.

In [ ]:
%pip install --upgrade airavata-python-sdk-test

## Importing the SDK

In [1]:
%cd dev-tools/deployment/jupyterhub/user-container/MD
import airavata_experiments as ae
from airavata_experiments import md

/Users/yasith/projects/artisan/airavata/dev-tools/deployment/jupyterhub/user-container/MD


using legacy validation callback


Authenticated via saved access token!


## Authenticating

To authenticate for remote execution, call the `ae.login()` method.
This method will prompt you to enter your credentials and authenticate your session.

In [2]:
ae.login()

Authenticated via saved access token!


Once authenticated, the `ae.list_runtimes()` function can be called to list HPC resources that the user has access to.

In [3]:
runtimes = ae.list_runtimes()
ae.display(runtimes)

,id,cluster,category,queue_name,node_count,cpu_count,walltime
0,remote,login.expanse.sdsc.edu,gpu,gpu-shared,1,10,30
1,remote,login.expanse.sdsc.edu,cpu,shared,1,10,30
2,remote,anvil.rcac.purdue.edu,cpu,shared,1,24,30


## Uploading Experiment Files

Drag and drop experiment files onto the workspace that this notebook is run on.

```bash
(sh) $: tree data
data
├── b4pull.pdb
├── b4pull.restart.coor
├── b4pull.restart.vel
├── b4pull.restart.xsc
├── par_all36_water.prm
├── par_all36m_prot.prm
├── pull.conf
├── structure.pdb
└── structure.psf

1 directory, 9 files

```

## Defining a NAMD Experiment

The `md.NAMD.initialize()` is used to define a NAMD experiment.
Here, provide the paths to the `.conf` file, the `.pdb` file, the `.psf` file, any optional files you want to run NAMD on.
You can preview the function definition through auto-completion.

```python
def initialize(
    name: str,
    config_file: str,
    pdb_file: str,
    psf_file: str,
    ffp_files: list[str],
    other_files: list[str] = [],
    parallelism: Literal['CPU', 'GPU'] = "CPU",
    num_replicas: int = 1
) -> Experiment[ExperimentApp]
```

To add replica runs, simply call the `exp.add_replica()` function.
You can call the `add_replica()` function as many times as you want replicas.
Any optional resource constraint can be provided here.

You can also call `ae.display()` to pretty-print the experiment.

In [4]:
exp = md.NAMD.initialize(
    name="yasith_namd_experiment",
    config_file="data/pull_cpu.conf",
    pdb_file="data/structure.pdb",
    psf_file="data/structure.psf",
    ffp_files=[
      "data/par_all36_water.prm",
      "data/par_all36m_prot.prm"
    ],
    other_files=[
      "data/b4pull.pdb",
      "data/b4pull.restart.coor",
      "data/b4pull.restart.vel",
      "data/b4pull.restart.xsc",
    ],
    parallelism="CPU",
    num_replicas=1,
)
exp.add_replica(*ae.list_runtimes(cluster="login.expanse.sdsc.edu", category="cpu"))
ae.display(exp)

Added replica. (1 tasks in total)


,name,application,num_tasks,config_file,pdb_file,psf_file,ffp_files,parallelism,other_files,num_replicas
0,yasith_namd_experiment,NAMD,1,data/pull_cpu.conf,data/structure.pdb,data/structure.psf,"data/par_all36_water.prm, data/par_all36m_prot...",CPU,"data/b4pull.pdb, data/b4pull.restart.coor, dat...",1


## Creating an Execution Plan

Call the `exp.plan()` function to transform the experiment definition + replicas into a stateful execution plan.

In [5]:
plan = exp.plan()
ae.display(plan)

,plan_id,name,app_id,inputs,runtime,ref,pid,agent_ref,workdir,sr_host
0,None,yasith_namd_experiment_DB23,NAMD,{'MD-Instructions-Input': {'value': 'data/pull...,"{'id': 'remote', 'args': {'cluster': 'login.ex...",None,None,None,None,None


## Saving the Plan

A created plan can be saved locally (in JSON) or remotely (in a user-local DB) for later reference.

In [6]:
plan.save()  # this will save the plan in DB
plan.save_json("plan.json")  # save the plan state locally

Plan saved: 8d7a6a06-e513-4c4e-8d9e-9da915e0690e


## Launching the Plan

A created plan can be launched using the `plan.launch()` function.
Changes to plan states will be automatically saved onto the remote.
However, plan state can also be tracked locally by invoking `plan.save_json()`.

In [7]:
plan.launch()
plan.save_json("plan.json")

Preparing execution plan...
Confirming execution plan...
Launching tasks...
[Task] Executing yasith_namd_experiment_DB23 on Remote(args={'cluster': 'login.expanse.sdsc.edu', 'category': 'cpu', 'queue_name': 'shared', 'node_count': 1, 'cpu_count': 10, 'walltime': 30})
[Remote] Creating Experiment: name=yasith_namd_experiment_DB23
[AV] Preprocessing args...
[AV] Validating args...
[AV] Setting up runtime params...
[AV] Setting up application interface...
[AV] Setting up experiment...
[AV] Setting up experiment directory...
[AV] exp_dir: /Default_Project/yasith_namd_experiment_DB23_2024_12_17_15_58_01
[AV] abs_path: /media/volume/cybershuttle-gateway-storage/pjaya001@odu.edu/Default_Project/yasith_namd_experiment_DB23_2024_12_17_15_58_01/
[AV] Setting up file inputs...
InputDataObjectType(name='agent_id', value=None, type=0, applicationArgument='-a', standardInput=False, userFriendlyDescription=None, metaData=None, inputOrder=14, isRequired=False, requiredToAddedToCommandLine=True, dataSt

Output()

[AV] Experiment yasith_namd_experiment_DB23 CREATED with id: yasith_namd_experiment_DB23_ce9ce6db-cb14-431d-a30b-2dffcf0b72e7
[AV] Experiment yasith_namd_experiment_DB23 STARTED with id: yasith_namd_experiment_DB23_ce9ce6db-cb14-431d-a30b-2dffcf0b72e7
[AV] Experiment yasith_namd_experiment_DB23 WAITING until experiment begins...
[AV] Experiment yasith_namd_experiment_DB23 EXECUTING with pid: PROCESS_41e7a453-346d-444b-a406-d63af89ba064
[Remote] Experiment Launched: id=yasith_namd_experiment_DB23_ce9ce6db-cb14-431d-a30b-2dffcf0b72e7
Plan updated: 8d7a6a06-e513-4c4e-8d9e-9da915e0690e


## Checking the Plan Status
The status of a plan can be retrieved by calling `plan.status()`.

In [8]:
plan.status()

Plan 8d7a6a06-e513-4c4e-8d9e-9da915e0690e (1 tasks):
* yasith_namd_experiment_DB23: EXECUTING


## Loading a Saved Plan

A saved plan can be loaded by calling `ae.plan.load_json(plan_path)` (for local plans) or `ae.plan.load(plan_id)` (for remote plans).

In [9]:
plan = ae.plan.load_json("plan.json")
plan = ae.plan.load(plan.id)
plan.status()
ae.display(plan)

Plan 8d7a6a06-e513-4c4e-8d9e-9da915e0690e (1 tasks):
* yasith_namd_experiment_DB23: EXECUTING


,plan_id,name,app_id,inputs,runtime,ref,pid,agent_ref,workdir,sr_host
0,8d7a6a06-e513-4c4e-8d9e-9da915e0690e,yasith_namd_experiment_DB23,NAMD,{'MD-Instructions-Input': {'value': 'data/pull...,"{'id': 'remote', 'args': {'cluster': 'login.ex...",yasith_namd_experiment_DB23_ce9ce6db-cb14-431d...,PROCESS_41e7a453-346d-444b-a406-d63af89ba064,818fa30a-d2ca-476f-9a1a-660473ae4f5c,/Default_Project/yasith_namd_experiment_DB23_2...,iguide-cybershuttle.che070035.projects.jetstre...


## Fetching User-Defined Plans

The `ae.plan.query()` function retrieves all plans stored in the remote.

In [10]:
plans = ae.plan.query()
ae.display(plans)

,plan_id,name,app_id,inputs,runtime,ref,pid,agent_ref,workdir,sr_host
0,8d7a6a06-e513-4c4e-8d9e-9da915e0690e,yasith_namd_experiment_DB23,NAMD,{'MD-Instructions-Input': {'value': 'data/pull...,"{'id': 'remote', 'args': {'cluster': 'login.ex...",yasith_namd_experiment_DB23_ce9ce6db-cb14-431d...,PROCESS_41e7a453-346d-444b-a406-d63af89ba064,818fa30a-d2ca-476f-9a1a-660473ae4f5c,/Default_Project/yasith_namd_experiment_DB23_2...,iguide-cybershuttle.che070035.projects.jetstre...
1,acaabb04-3876-4c73-90d9-265f0984c09a,yasith_namd_experiment_BB6D,NAMD,{'MD-Instructions-Input': {'value': 'data/pull...,"{'id': 'remote', 'args': {'cluster': 'login.ex...",yasith_namd_experiment_BB6D_5365a12c-0b42-480f...,PROCESS_cff13fe6-5c11-4915-8be7-25da84edbaf7,ddbfe374-83e2-4151-bd02-cbd7fb676950,/Default_Project/yasith_namd_experiment_BB6D_2...,iguide-cybershuttle.che070035.projects.jetstre...


## Managing Plan Execution

The `plan.stop()` function will stop a currently executing plan.
The `plan.wait_for_completion()` function would block until the plan finishes executing.

In [11]:
# plan.stop()
plan.wait_for_completion()

Output()

Interrupted by user.


## Interacting with Files

The `task` object has several helper functions to perform file operations within its context.

* `task.ls()` - list all remote files (inputs, outputs, logs, etc.)
* `task.upload(<local_path>, <remote_path>)` - upload a local file to remote
* `task.cat(<remote_path>)` - displays contents of a remote file
* `task.download(<remote_path>, <local_path>)` - fetch a remote file to local

In [12]:
for task in plan.tasks:
    print(task.name, task.pid)
    # display files
    display(task.ls())
    # upload a file
    task.upload("data/sample.txt")
    # preview contents of a file
    display(task.cat("sample.txt"))
    # download a specific file
    task.download("sample.txt", f"./results_{task.name}")
    # download all files
    task.download_all(f"./results_{task.name}")

yasith_namd_experiment_DB23 PROCESS_41e7a453-346d-444b-a406-d63af89ba064


['pull_cpu.conf',
 'structure.pdb',
 'structure.psf',
 'par_all36_water.prm',
 'par_all36m_prot.prm',
 'b4pull.pdb',
 'b4pull.restart.coor',
 'b4pull.restart.vel',
 'b4pull.restart.xsc']

Output()

b'hello from the sample file!'

Output()

## Executing Task-Local Code Remotely

The `@task.context()` decorator can be applied on Python functions to run them remotely within the task context.
The functions executed this way has access to the task files, as well as the remote compute resources.

**NOTE**: Currently, remote code execution is only supported for ongoing tasks. In future updates, we will support both ongoing and completed tasks. Stay tuned!

In [ ]:
for index, task in enumerate(plan.tasks):
    @task.context(packages=["numpy", "pandas"])
    def analyze() -> None:
        import numpy as np
        with open("pull.conf", "r") as f:
            data = f.read()
        print("pull.conf has", len(data), "chars")
        print(np.arange(10))
    analyze()